# 04 — Baseline-Modelle & Ablationsstudie

**Ziel dieses Notebooks:** Drei klassische Modelle (Logistic Regression,
Naive Bayes, Linear SVC) auf vier Vektorisierungs-Konfigurationen aus
Notebook 03 trainieren und evaluieren — systematische Ablationsstudie,
welche Kombination den besten F1-Score liefert.

**Leitfragen:**
- Welches Modell performt am besten auf diesem Datensatz?
- Welche Vektorisierungs-Konfiguration (A/B/C/D) ist optimal?
- Verbessern Bigramme (Config B) den F1 gegenueber Unigrammen (Config A)?
- Bringt LSA (Config D) einen Vorteil gegenueber direktem TF-IDF?
- Liefert keyword als zusaetzliches Feature einen messbaren Gewinn?
- Wie weit liegt das beste Ergebnis vom realistischen SOTA (F1 ~0,83-0,85)
  entfernt, und warum?

**Methodik:**
- Stratified K-Fold CV (k=5) als Pflicht — verhindert Data Leakage,
  behaelt Klassenbalance pro Fold
- Hauptmetrik: F1 (Mittelwert ± Standardabweichung ueber 5 Folds)
- Zusaetzlich: ROC-AUC, Precision-Recall-Kurve (beste Kombination)
- 4 Konfigurationen x 3 Modelle = 12 Kombinationen

**Output:** Modellvergleich-Tabelle, beste Modelle gespeichert als
.joblib, Fehleranalyse-Vorbereitung fuer Phase 6.

In [ ]:
# =============================================================================
# Zelle 02 – Imports, Matrizen + Vectorizer laden, Store44-Style aktivieren
# =============================================================================

import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from viz_config import (
    apply_store44_style, save_figure,
    COLOR_GOLD, COLOR_BLUE, COLOR_GREEN,
    COLOR_CLASS_0, COLOR_CLASS_1, COLOR_TEXT_MUTED
)

apply_store44_style()

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import f1_score, roc_auc_score, make_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

# Datensatz laden
train = pd.read_csv(
    "../data/processed/train_preprocessed.csv",
    keep_default_na=False,
)
train[["keyword", "location"]] = train[["keyword", "location"]].replace("", np.nan)
y = train["target"].values

print("Shape train:", train.shape)
print("Klassenverteilung:", np.bincount(y))

# Vektorisierungs-Konfigurationen neu aufbauen
# (Matrizen nicht direkt geladen, da sparse Matrizen nicht direkt als .npy
# gespeichert wurden — Vectorizer sind gespeichert, Matrix wird hier
# reproduziert fuer volle Kontrolle/Transparenz)

vectorizer_A = joblib.load("../models/vectorizer_A.joblib")
vectorizer_B = joblib.load("../models/vectorizer_B.joblib")
vectorizer_C = joblib.load("../models/vectorizer_C.joblib")
vectorizer_D_bundle = joblib.load("../models/vectorizer_D_lsa.joblib")

X_A = vectorizer_A.transform(train["text_no_stopwords"])
X_B = vectorizer_B.transform(train["text_no_stopwords"])
X_C = vectorizer_C.transform(train["text_stemmed"])
X_D = vectorizer_D_bundle["normalizer"].transform(
    vectorizer_D_bundle["svd"].transform(
        vectorizer_D_bundle["vectorizer"].transform(train["text_no_stopwords"])
    )
)

configs = {"A": X_A, "B": X_B, "C": X_C, "D": X_D}

print("\nKonfigurationen geladen:")
for name, X in configs.items():
    print(f"  Config {name}: {X.shape}")

In [ ]:
# =============================================================================
# Zelle 03 – Stratified K-Fold CV: vollstaendige Evaluationsfunktion
# =============================================================================
# Metriken: F1 (Macro + per Klasse), Precision_1, Recall_1, ROC-AUC,
#           MCC, Train-CV-Gap (Overfitting), Bootstrap-KI (95%)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score,
    recall_score, matthews_corrcoef
)
import warnings

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(model, X, y, model_name, config_name,
                   skf=SKF, n_bootstrap=500):

    fold_metrics = {
        "f1_macro": [], "f1_0": [], "f1_1": [],
        "precision_1": [], "recall_1": [],
        "roc_auc": [], "mcc": [],
        "f1_macro_train": []
    }

    # Out-of-fold Vorhersagen sammeln (fuer korrektes Bootstrap)
    oof_preds = np.zeros(len(y), dtype=int)
    oof_scores = np.zeros(len(y))

    for train_idx, val_idx in skf.split(X, y):
        if hasattr(X, "toarray"):
            X_tr, X_val = X[train_idx], X[val_idx]
        else:
            X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        y_score = (model.predict_proba(X_val)[:, 1]
                   if hasattr(model, "predict_proba")
                   else model.decision_function(X_val))

        # Out-of-fold speichern
        oof_preds[val_idx] = y_pred
        oof_scores[val_idx] = y_score

        fold_metrics["f1_macro"].append(
            f1_score(y_val, y_pred, average="macro"))
        fold_metrics["f1_0"].append(
            f1_score(y_val, y_pred, pos_label=0))
        fold_metrics["f1_1"].append(
            f1_score(y_val, y_pred, pos_label=1))
        fold_metrics["precision_1"].append(
            precision_score(y_val, y_pred, zero_division=0))
        fold_metrics["recall_1"].append(
            recall_score(y_val, y_pred, zero_division=0))
        fold_metrics["roc_auc"].append(
            roc_auc_score(y_val, y_score))
        fold_metrics["mcc"].append(
            matthews_corrcoef(y_val, y_pred))

        # Train-Score fuer Overfitting-Gap
        y_pred_train = model.predict(X_tr)
        fold_metrics["f1_macro_train"].append(
            f1_score(y_tr, y_pred_train, average="macro"))

    results = {"model": model_name, "config": config_name}
    for metric, values in fold_metrics.items():
        results[f"{metric}_mean"] = np.mean(values)
        results[f"{metric}_std"] = np.std(values)

    results["overfitting_gap"] = (
        results["f1_macro_train_mean"] - results["f1_macro_mean"])

    # Bootstrap auf OUT-OF-FOLD Vorhersagen (kein Datenleck)
    rng = np.random.RandomState(42)
    boot_f1, boot_roc = [], []
    for _ in range(n_bootstrap):
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        boot_f1.append(f1_score(y[idx], oof_preds[idx], average="macro"))
        boot_roc.append(roc_auc_score(y[idx], oof_scores[idx]))

    results["f1_macro_ci_low"] = np.percentile(boot_f1, 2.5)
    results["f1_macro_ci_high"] = np.percentile(boot_f1, 97.5)
    results["roc_auc_ci_low"] = np.percentile(boot_roc, 2.5)
    results["roc_auc_ci_high"] = np.percentile(boot_roc, 97.5)

    return results


print("Evaluationsfunktion bereit.")
print("Metriken: F1 Macro/Klasse, Precision_1, Recall_1, ROC-AUC, MCC")
print("Zusatz:   Bootstrap-KI (95%, n=500), Overfitting-Gap (Train vs. CV)")
print(f"K-Fold:   k=5, shuffle=True, random_state=42")

## Methodik: Stratified K-Fold Cross-Validation (k=5)

**Warum Cross-Validation statt einfachem Train/Test-Split?**
Ein einzelner Train/Test-Split haengt stark davon ab, welche Zeilen
zufaellig ins Test-Set landen — bei 6.801 Tweets entspricht 1
falsch klassifizierter Tweet ~0,01 F1-Aenderung. Das Ergebnis waere
statistisch instabil.

**K-Fold Prinzip:**
Der Datensatz wird in k=5 gleich grosse Teile (Folds) aufgeteilt.
In 5 Durchlaeufen wird jeder Fold einmal als Validierungsset verwendet,
die restlichen 4 als Trainingsset. Ergebnis: 5 unabhaengige F1-Scores
→ Mittelwert ± Standardabweichung.

**Warum Stratified?**
Ohne Stratifizierung koennte ein Fold zufaellig mehr Disaster-Tweets
enthalten als ein anderer (60/40-Verteilung im Datensatz). Stratified
K-Fold garantiert, dass jeder Fold dieselbe Klassenverteilung behaelt
(59,4% / 40,6%).

**Was die Standardabweichung ueber Folds aussagt:**
- Niedrige Std (< 0,01): stabile, zuverlaessige Leistung
- Hohe Std (> 0,03): Modell reagiert sensibel auf Datensplit —
  Hinweis auf Instabilitaet oder Ueberanpassung

**Overfitting-Gap:**
Train-F1 minus CV-F1 pro Fold. Kleiner Gap (~0,01-0,02) = gute
Generalisierung. Grosser Gap (> 0,05) = Modell lernt Trainingsdaten
auswendig, verallgemeinert schlecht auf neue Daten.

**Bootstrap-Konfidenzintervalle (95%, n=500):**
Quantifiziert die statistische Unsicherheit der Punkt-Schaetzung —
zeigt, in welchem Bereich der wahre F1 mit 95% Wahrscheinlichkeit
liegt. Breite KI = hohe Unsicherheit (wenig Daten), schmale KI =
stabile Schaetzung.

In [ ]:
# =============================================================================
# Zelle 05 – Logistic Regression: alle Konfigurationen + class_weight Ablation
# =============================================================================

from sklearn.linear_model import LogisticRegression

logreg_results = []

logreg_variants = [
    ("LogReg", LogisticRegression(max_iter=1000, random_state=42)),
    ("LogReg_balanced", LogisticRegression(
        max_iter=1000, random_state=42, class_weight="balanced")),
]

for model_name, model in logreg_variants:
    for config_name, X in configs.items():
        print(f"Evaluiere {model_name} | Config {config_name}...", end=" ")
        result = evaluate_model(model, X, y, model_name, config_name)
        logreg_results.append(result)
        print(f"F1={result['f1_macro_mean']:.4f} | Gap={result['overfitting_gap']:.4f}")

logreg_df = pd.DataFrame(logreg_results)

# Kompakte Ausgabe der besten Kombination
best_logreg = logreg_df.loc[logreg_df["f1_macro_mean"].idxmax()]
print(f"\nBeste LogReg-Kombination:")
print_result(best_logreg)

# Export
logreg_df.to_csv("../reports/tables/04_logreg_results.csv", index=False)

## Erkenntnis: Logistic Regression

| Config | Modell | F1 Macro | Gap | F1 Klasse 0 | F1 Klasse 1 |
|---|---|---|---|---|---|
| A | LogReg | 0.7730 | 0.093 | — | — |
| B | LogReg | 0.7727 | 0.102 | — | — |
| C | LogReg | 0.7746 | 0.084 | — | — |
| D | LogReg | 0.7506 | 0.011 | — | — |
| A | LogReg_balanced | **0.7776** | 0.108 | 0.823 | 0.732 |
| B | LogReg_balanced | 0.7771 | 0.115 | — | — |
| C | LogReg_balanced | 0.7775 | 0.099 | — | — |
| D | LogReg_balanced | 0.7396 | 0.014 | — | — |

**Beste Kombination:** LogReg_balanced + Config A
  F1 Macro: 0.7776 ± 0.0098 [95% KI: 0.768 – 0.788]
  ROC-AUC:  0.8464 ± 0.0123 [95% KI: 0.837 – 0.857]
  MCC:      0.5557 ± 0.0191

**Kernbefunde:**

1. **class_weight="balanced" bringt marginalen Gewinn (+0.003-0.005 F1)**
   aber erhoehten Overfitting-Gap (+0.015-0.014). Tradeoff: leicht
   bessere CV-Performance auf Kosten staerkerer Ueberanpassung.

2. **Config D (LSA) faellt klar ab** (F1 0.750/0.740 vs. 0.773-0.778)
   — bestaetigt die Erwartung aus Notebook 03: 13,8% erklaerte Varianz
   sind zu wenig Signal fuer gute Klassifikation. Positiv: fast kein
   Overfitting-Gap (0.011/0.014) — weniger Dimensionen = weniger
   Ueberanpassungsrisiko, aber auch weniger Trennkraft.

3. **Config C (Stemmed) minimal besser als A und B** (0.7746 vs.
   0.7730/0.7727) ohne class_weight — kleinstes Vokabular (4.869)
   scheint bei LogReg leicht vorteilhaft (weniger Rauschen durch
   Zusammenfuehrung von Wortformen).

4. **Asymmetrie zwischen Klassen:** F1 Klasse 0 (0.823) deutlich
   besser als F1 Klasse 1 (0.732) — Modell erkennt "kein Disaster"
   zuverlaessiger als echte Disasters. Bestaetigt den Kontext-Befund
   aus Notebook 01: Disaster-Vokabular wird haeufig metaphorisch
   verwendet, was False Positives erzeugt und Recall_1 (0.718) senkt.

5. **Overfitting-Gap bei A/B/C: 0.084-0.115** — nicht kritisch fuer
   ein lineares Modell, aber messbar. Train-F1 (0.886) deutlich ueber
   CV-F1 (0.778) deutet darauf hin, dass das Modell das Trainingsset
   besser "auswendig lernt" als es generalisiert. Wird in Phase 5
   (Tuning) adressiert.

**SOTA-Einordnung:**
  LogReg beste F1: 0.7776 vs. realistischer SOTA: 0.83-0.85
  Abstand: ~0.05-0.07 F1-Punkte

In [ ]:
# =============================================================================
# Zelle 07 – Naive Bayes: alle Konfigurationen
# =============================================================================
# Zwei Varianten:
# - MultinomialNB: klassisch fuer Wortzaehlungen/TF-IDF (erwartet >= 0)
# - ComplementNB:  verbesserte Variante, oft besser bei unbalancierten Klassen
# Kein class_weight verfuegbar bei NB — ComplementNB ist die NB-eigene
# Antwort auf Klassenungleichgewicht.
# Hinweis: Config D (LSA) enthaelt negative Werte nach Normalisierung
# — MultinomialNB/ComplementNB nicht anwendbar, wird uebersprungen.

from sklearn.naive_bayes import MultinomialNB, ComplementNB

nb_results = []

nb_variants = [
    ("MultinomialNB", MultinomialNB()),
    ("ComplementNB", ComplementNB()),
]

for model_name, model in nb_variants:
    for config_name, X in configs.items():
        # Config D (LSA) enthaelt negative Werte -> NB nicht anwendbar
        if config_name == "D":
            print(f"  {model_name} | Config D: uebersprungen (negative Werte in LSA)")
            continue
        print(f"Evaluiere {model_name} | Config {config_name}...", end=" ")
        result = evaluate_model(model, X, y, model_name, config_name)
        nb_results.append(result)
        print(f"F1={result['f1_macro_mean']:.4f} | Gap={result['overfitting_gap']:.4f}")

nb_df = pd.DataFrame(nb_results)

# Beste Kombination
best_nb = nb_df.loc[nb_df["f1_macro_mean"].idxmax()]
print(f"\nBeste NB-Kombination:")
print_result(best_nb)

nb_df.to_csv("../reports/tables/04_nb_results.csv", index=False)

## Erkenntnis: Naive Bayes

| Config | Modell | F1 Macro | Gap | Precision_1 | Recall_1 |
|---|---|---|---|---|---|
| A | MultinomialNB | 0.7781 | 0.086 | — | — |
| B | MultinomialNB | 0.7743 | 0.087 | — | — |
| C | MultinomialNB | **0.7799** | **0.077** | 0.825 | 0.636 |
| D | MultinomialNB | n/a | n/a | — | — |
| A | ComplementNB | 0.7705 | 0.105 | — | — |
| B | ComplementNB | 0.7758 | 0.109 | — | — |
| C | ComplementNB | 0.7704 | 0.095 | — | — |
| D | ComplementNB | n/a | n/a | — | — |

**Beste Kombination:** MultinomialNB + Config C (Stemmed)
  F1 Macro:    0.7799 ± 0.0100 [95% KI: 0.770 – 0.789]
  ROC-AUC:     0.8446 ± 0.0081 [95% KI: 0.836 – 0.856]
  MCC:         0.5757 ± 0.0181
  Overfitting-Gap: 0.077 (niedrigster aller bisherigen Modelle)

**Kernbefunde:**

1. **ComplementNB NICHT besser als MultinomialNB** — entgegen der
   Erwartung ist MultinomialNB in allen drei Konfigurationen staerker
   (0.778-0.780 vs. 0.770-0.776). ComplementNB zeigt auch hoeheren
   Overfitting-Gap. Moegliche Erklaerung: das Klassenungleichgewicht
   (60/40) ist zu moderat fuer ComplementNB's Staerke — diese Variante
   profitiert staerker bei extremeren Verhaeltnissen (z.B. 90/10).

2. **Config C (Stemmed) ist erneut beste Text-Variante** — bereits
   bei LogReg beobachtet, hier bestaetigt. Kleineres Vokabular durch
   Stammformreduktion scheint bei wortbasierter Naive Bayes Klassifikation
   vorteilhaft (weniger Dimensionen, weniger Rauschen durch Wortform-
   Varianten wie "fires/fire/fired" → "fire").

3. **Praezisionsstarkes Profil:** Precision_1 (0.825) deutlich hoeher
   als Recall_1 (0.636) — MultinomialNB ist sehr konservativ bei der
   Disaster-Vorhersage: trifft wenn, dann oft richtig, verpasst aber
   viele echte Disasters. In einem realen Anwendungsfall (Disaster-
   Response) waere dieses Profil problematisch (hohe FN-Rate).

4. **Niedrigster Overfitting-Gap aller bisherigen Modelle (0.077)**
   — konsistent mit der NB-Eigenschaft: keine iterative Optimierung,
   direkte Wahrscheinlichkeitsschaetzung aus Haeufigkeiten. Modell
   "lernt" nicht ueberangepasst.

5. **Config D (LSA) nicht kompatibel** mit NB — negative Werte
   nach SVD-Normalisierung verletzen die NB-Grundannahme
   (Wahrscheinlichkeiten muessen >= 0 sein). Architekturelle
   Inkompatibilitaet, kein Fehler.

**Vergleich mit LogReg:**
  MultinomialNB F1: 0.7799 vs. LogReg_balanced F1: 0.7776
  NB minimal besser (+0.002), aber mit anderem Fehler-Profil:
  NB hat hoeheres Precision_1 (0.825 vs. 0.747) aber niedrigeres
  Recall_1 (0.636 vs. 0.718). NB verpasst mehr Disasters, macht
  aber weniger Fehlalarme.

In [ ]:
# =============================================================================
# Zelle 09 – Linear SVC: alle Konfigurationen + class_weight Ablation
# =============================================================================
# LinearSVC: optimiert direkt den Margin (Abstand zur Trennebene) —
# theoretisch gut geeignet fuer hochdimensionale, sparse Daten (TF-IDF).
# Kein predict_proba() verfuegbar — ROC-AUC wird ueber decision_function()
# berechnet (Score statt Wahrscheinlichkeit, aber fuer Ranking ausreichend).

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svc_results = []

svc_variants = [
    ("LinearSVC", LinearSVC(max_iter=2000, random_state=42)),
    ("LinearSVC_balanced", LinearSVC(
        max_iter=2000, random_state=42, class_weight="balanced")),
]

for model_name, model in svc_variants:
    for config_name, X in configs.items():
        print(f"Evaluiere {model_name} | Config {config_name}...", end=" ")
        result = evaluate_model(model, X, y, model_name, config_name)
        svc_results.append(result)
        print(f"F1={result['f1_macro_mean']:.4f} | Gap={result['overfitting_gap']:.4f}")

svc_df = pd.DataFrame(svc_results)

# Beste Kombination
best_svc = svc_df.loc[svc_df["f1_macro_mean"].idxmax()]
print(f"\nBeste SVC-Kombination:")
print_result(best_svc)

svc_df.to_csv("../reports/tables/04_svc_results.csv", index=False)

## Erkenntnis: Linear SVC

| Config | Modell | F1 Macro | Gap | Precision_1 | Recall_1 |
|---|---|---|---|---|---|
| A | LinearSVC | 0.7652 | 0.195 | — | — |
| B | LinearSVC | 0.7658 | 0.206 | — | — |
| C | LinearSVC | **0.7681** | 0.181 | 0.754 | 0.682 |
| D | LinearSVC | 0.7500 | 0.009 | — | — |
| A | LinearSVC_balanced | 0.7576 | 0.208 | — | — |
| B | LinearSVC_balanced | 0.7579 | 0.218 | — | — |
| C | LinearSVC_balanced | 0.7603 | 0.194 | — | — |
| D | LinearSVC_balanced | 0.7406 | 0.014 | — | — |

**Beste Kombination:** LinearSVC + Config C (Stemmed)
  F1 Macro:        0.7681 ± 0.0069 [95% KI: 0.759 – 0.778]
  ROC-AUC:         0.8302 ± 0.0121 [95% KI: 0.820 – 0.841]
  MCC:             0.5393 ± 0.0120
  Overfitting-Gap: 0.181 (hoechster aller drei Modelle)
  Train-F1:        0.949

**Kernbefunde:**

1. **Ueberraschendes Ergebnis: LinearSVC ist das schwaechste Modell**
   (F1 0.768 vs. NB 0.780, LogReg 0.778) — entgegen der theoretischen
   Erwartung (SVC gilt als stark bei hochdimensionalen sparse Daten).
   Ursache: starkes Overfitting (Train-F1 0.949, Gap 0.181-0.218).

2. **Bias-Variance Tradeoff — Diagnose:**
   LinearSVC optimiert direkt den Margin und kann bei 5.838 Dimensionen
   und 6.801 Datenpunkten eine Hyperebene finden, die Trainingsdaten
   fast perfekt trennt (Train-F1 0.949). Diese Hyperebene ist zu
   spezifisch fuer die Trainingsdaten — sie generalisiert schlecht.
   Das ist ein klassisches High-Variance-Problem.

   Moegliche Loesung in Phase 5 (Tuning): Regularisierungsparameter C
   reduzieren (staerkere Regularisierung = kleinerer Margin = weniger
   Ueberanpassung). C ist der wichtigste Hyperparameter fuer SVC.

3. **class_weight="balanced" schadet hier** (-0.008 bis -0.010 F1) —
   bei SVC verschiebt Balancing die Entscheidungsgrenze so, dass die
   Minority-Klasse (Disaster) bevorzugt wird, was bei diesem Datensatz
   (60/40, kein extremes Ungleichgewicht) die Gesamtperformance senkt.

4. **Config D (LSA, Gap=0.009) bestaetigt das Muster:** Wenige
   Dimensionen = wenig Overfitting, aber auch wenig Signal. LinearSVC
   profitiert stark von mehr Regularisierung — Phase 5 wird zeigen,
   ob ein gut getuntes LinearSVC die anderen Modelle schlagen kann.

5. **Niedrigste Std aller Modelle (F1 ± 0.007):** LinearSVC ist
   stabiler ueber Folds als LogReg (±0.010) und NB (±0.010) — die
   Vorhersagen schwanken weniger je nach Datensplit, auch wenn das
   absolute Niveau niedriger ist.

**Konsequenz fuer Phase 5 (Tuning):**
LinearSVC ist der vielversprechendste Tuning-Kandidat: grosser
Overfitting-Gap zeigt, dass Performance durch Regularisierung (C)
noch erheblich verbessert werden kann. NB hat wenig
Tuning-Potenzial (fast kein Gap), LogReg ist Mittelweg.

In [ ]:
# =============================================================================
# Zelle 11 – Gesamtvergleich: alle Modelle x Konfigurationen
# =============================================================================

# Alle Ergebnisse zusammenfuehren
all_results_df = pd.concat([logreg_df, nb_df, svc_df], ignore_index=True)
all_results_df.to_csv("../reports/tables/04_all_results.csv", index=False)

print("=== Gesamtuebersicht (sortiert nach F1 Macro) ===")
summary = all_results_df[["model", "config", "f1_macro_mean", "f1_macro_std",
                           "f1_0_mean", "f1_1_mean", "precision_1_mean",
                           "recall_1_mean", "roc_auc_mean", "mcc_mean",
                           "overfitting_gap"]].copy()
summary = summary.sort_values("f1_macro_mean", ascending=False)
pd.set_option("display.float_format", "{:.4f}".format)
print(summary.to_string(index=False))

# Top-3 Kombinationen
print("\n=== Top 3 Kombinationen ===")
for _, row in summary.head(3).iterrows():
    print(f"  {row['model']:25s} | Config {row['config']} | "
          f"F1={row['f1_macro_mean']:.4f} | "
          f"ROC-AUC={row['roc_auc_mean']:.4f} | "
          f"MCC={row['mcc_mean']:.4f} | "
          f"Gap={row['overfitting_gap']:.4f}")

# Plot: F1 Macro nach Modell und Konfiguration
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

model_groups = [
    (["LogReg", "LogReg_balanced"], "Logistic Regression", axes[0]),
    (["MultinomialNB", "ComplementNB"], "Naive Bayes", axes[1]),
    (["LinearSVC", "LinearSVC_balanced"], "Linear SVC", axes[2]),
]

colors_config = {"A": COLOR_GOLD, "B": COLOR_BLUE,
                 "C": COLOR_GREEN, "D": COLOR_TEXT_MUTED}

for model_names, title, ax in model_groups:
    subset = all_results_df[all_results_df["model"].isin(model_names)]
    x = np.arange(len(model_names))
    width = 0.2
    offsets = [-1.5, -0.5, 0.5, 1.5]

    for i, config in enumerate(["A", "B", "C", "D"]):
        config_data = subset[subset["config"] == config]
        if config_data.empty:
            continue
        vals = config_data["f1_macro_mean"].values
        errs = config_data["f1_macro_std"].values
        positions = [x[j] + offsets[i] * width
                     for j in range(len(model_names))
                     if j < len(vals)]
        ax.bar(positions, vals, width,
               color=colors_config[config],
               alpha=0.85, label=f"Config {config}",
               yerr=errs, capsize=3,
               error_kw={"ecolor": COLOR_TEXT_MUTED, "elinewidth": 1.5})

    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace("_balanced", "\n(balanced)")
                        for m in model_names], fontsize=9)
    ax.set_ylabel("F1 Macro (CV)")
    ax.set_ylim(0.72, 0.82)
    ax.axhline(0.7776, color=COLOR_GOLD, linestyle="--",
               alpha=0.5, linewidth=1)

# Legende nur einmal
handles = [plt.Rectangle((0, 0), 1, 1, color=colors_config[c])
           for c in ["A", "B", "C", "D"]]
fig.legend(handles, ["Config A (TF-IDF Unigram)",
                     "Config B (TF-IDF Bigram)",
                     "Config C (TF-IDF Stemmed)",
                     "Config D (LSA 100D)"],
           loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.05))

fig.suptitle("Modellvergleich: F1 Macro (5-Fold CV) — alle Kombinationen\n"
             "Gestrichelte Linie = beste Kombination (LogReg_balanced Config A)")
fig.tight_layout(rect=[0, 0.05, 1, 1])

save_figure(fig, "../reports/figures/04_model_comparison.png")
plt.show()

## Erkenntnis: Gesamtvergleich — alle Modelle & Konfigurationen

### Rangliste (Top 5 nach F1 Macro)

| Rang | Modell | Config | F1 Macro | Std | ROC-AUC | MCC | Gap |
|---|---|---|---|---|---|---|---|
| 1 | MultinomialNB | C | 0.7799 | 0.0100 | 0.8446 | 0.5757 | 0.077 |
| 2 | MultinomialNB | A | 0.7781 | 0.0119 | 0.8439 | 0.5725 | 0.086 |
| 3 | LogReg_balanced | A | 0.7776 | 0.0098 | 0.8464 | 0.5557 | 0.108 |
| 4 | LogReg_balanced | C | 0.7775 | **0.0069** | **0.8487** | 0.5551 | 0.099 |
| 5 | LogReg_balanced | B | 0.7771 | 0.0105 | 0.8449 | 0.5548 | 0.115 |

**Entscheidung: LogReg_balanced Config C als Hauptkandidat Phase 5**

Begruendung: F1-Unterschied zu MultinomialNB (0.0024) liegt innerhalb
beider Standardabweichungen — statistisch nicht signifikant. LogReg
ueberzeugt durch:
- Niedrigste Std aller Top-5 (0.0069) — stabiler, vorhersehbarer
- Hoechster ROC-AUC (0.8487) — bestes Ranking-Verhalten
- Ausgewogeneres Fehler-Profil (Precision_1=0.741, Recall_1=0.728)
  vs. NB (Precision_1=0.825, Recall_1=0.636)
- Besser interpretierbar (Koeffizienten pro Wort direkt lesbar)
- Mehr Tuning-Hebel (C, solver, penalty) fuer Phase 5

MultinomialNB Config C als Vergleichskandidat behalten.

### Systemuebergreifende Erkenntnisse

**Config C (Stemmed) konsistent beste Text-Variante:**
In allen Modellen liegt Config C vorne oder gleichauf — Stemming
reduziert Wortform-Varianten (fires/fire/fired → fire), was bei
allen drei Modell-Typen zu konsistent besserem Signal fuehrt.

**Config D (LSA) durchgaengig schlechteste Performance:**
LSA 100D erklaert nur 13,8% Varianz des TF-IDF-Raums — zu wenig
Signal fuer gute Klassifikation. Einziger Vorteil: minimaler
Overfitting-Gap (0.009-0.014). Wird nicht weiter verfolgt.

**Fehler-Profile unterscheiden sich systematisch:**
- NB-Profil: konservativ (hohe Precision, niedriger Recall)
  → wenige Fehlalarme, aber viele verpasste Disasters
- LogReg-Profil: ausgewogen (moderate Precision und Recall)
  → mehr Fehlalarme, aber weniger verpasste Disasters
- SVC-Profil: aehnlich LogReg, aber mit starkem Overfitting

Welches Profil "besser" ist haengt vom Anwendungsfall ab:
Im realen Disaster-Response-Kontext waere LogReg vorzuziehen
(FN = verpasste Katastrophe ist teurer als FP = Fehlalarm).

**LinearSVC: groesstes Tuning-Potenzial, aktuell schwachste Performance:**
Gap 0.18-0.22 zeigt systematisches Overfitting — Regularisierungs-
parameter C in Phase 5 koennte LinearSVC noch deutlich verbessern.

**SOTA-Abstand:**
Beste Baseline F1: 0.7799 vs. SOTA F1: 0.83-0.85
Abstand: ~0.05-0.07 F1-Punkte — Hauptursachen:
1. Kontext-Blindheit von TF-IDF (Metaphern nicht erkannt)
2. Begrenzte Datenmenge (6.801 Tweets)
3. Ungetuntes Modell (Phase 5 erwartet +0.01-0.03 F1)

In [ ]:
# =============================================================================
# Zelle 13 – ROC-AUC + Precision-Recall-Kurve (beste Kombination)
# =============================================================================
# Visualisierung fuer LogReg_balanced Config C (Hauptkandidat Phase 5)
# und MultinomialNB Config C (Vergleichskandidat) — beide auf Out-of-Fold
# Vorhersagen, damit kein Datenleck.

from sklearn.metrics import (
    roc_curve, precision_recall_curve, auc,
    average_precision_score
)

# Out-of-Fold Scores fuer beide Kandidaten sammeln
candidates = [
    ("LogReg_balanced Config C",
     LogisticRegression(max_iter=1000, random_state=42,
                        class_weight="balanced"),
     configs["C"]),
    ("MultinomialNB Config C",
     MultinomialNB(),
     configs["C"]),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors_candidates = [COLOR_GOLD, COLOR_BLUE]

for (name, model, X), color in zip(candidates, colors_candidates):
    # Out-of-Fold Scores
    oof_scores = np.zeros(len(y))
    for train_idx, val_idx in SKF.split(X, y):
        if hasattr(X, "toarray"):
            X_tr, X_val = X[train_idx], X[val_idx]
        else:
            X_tr, X_val = X[train_idx], X[val_idx]
        model.fit(X_tr, y[train_idx])
        if hasattr(model, "predict_proba"):
            oof_scores[val_idx] = model.predict_proba(X_val)[:, 1]
        else:
            oof_scores[val_idx] = model.decision_function(X_val)

    # ROC-Kurve
    fpr, tpr, _ = roc_curve(y, oof_scores)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f"{name} (AUC={roc_auc:.4f})")

    # Precision-Recall-Kurve
    precision, recall, _ = precision_recall_curve(y, oof_scores)
    pr_auc = average_precision_score(y, oof_scores)
    axes[1].plot(recall, precision, color=color, linewidth=2,
                 label=f"{name} (AUC={pr_auc:.4f})")

# ROC-Plot
axes[0].plot([0, 1], [0, 1], color=COLOR_TEXT_MUTED,
             linestyle="--", linewidth=1, label="Zufall (AUC=0.500)")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC-Kurve (Out-of-Fold)")
axes[0].legend(fontsize=9)

# PR-Plot
baseline_pr = y.mean()
axes[1].axhline(baseline_pr, color=COLOR_TEXT_MUTED,
                linestyle="--", linewidth=1,
                label=f"Zufall (Baseline={baseline_pr:.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall-Kurve (Out-of-Fold)")
axes[1].legend(fontsize=9)

fig.suptitle("ROC-AUC & Precision-Recall: LogReg_balanced vs. MultinomialNB (Config C)")
fig.tight_layout()

save_figure(fig, "../reports/figures/04_roc_pr_curves.png")
plt.show()

## Erkenntnis: ROC-AUC & Precision-Recall-Kurven

### ROC-Kurve
| Modell | ROC-AUC |
|---|---|
| LogReg_balanced Config C | **0.8486** |
| MultinomialNB Config C | 0.8447 |
| Zufall | 0.500 |

Beide Kurven verlaufen nah beieinander — LogReg liegt konsistent
leicht ueber NB, besonders im mittleren FPR-Bereich (0.2-0.6).
Beide Modelle trennen die Klassen deutlich besser als Zufall.

### Precision-Recall-Kurve
| Modell | PR-AUC |
|---|---|
| MultinomialNB Config C | **0.8305** |
| LogReg_balanced Config C | 0.8232 |
| Zufall (Baseline) | 0.406 |

**Kreuzung der Kurven — wichtigster visueller Befund:**
- Bei niedrigem Recall (0.0-0.5): NB hat hoehere Precision
  → NB ist praeziser, wenn wenige Disasters vorhergesagt werden
- Bei hohem Recall (> 0.5): LogReg hat hoehere Precision
  → LogReg ist praeziser, wenn viele/alle Disasters erkannt werden sollen

Das ist die geometrische Bestaetigung des Fehler-Profil-Unterschieds:
NB = konservativ (vorzuziehen wenn FP teuer), LogReg = ausgewogen
(vorzuziehen wenn FN teuer, z. B. Disaster-Response).

**Threshold-Optimierung:** Die PR-Kurve zeigt direkt, welcher
Schwellenwert welche Precision/Recall-Kombination liefert. Standard-
schwellenwert (0.5) ist nicht zwingend optimal — wird in Phase 5
als Tuning-Parameter beruecksichtigt.

**Einordnung ggue. SOTA:**
PR-AUC beste Baseline: 0.8305 vs. erwarteter SOTA: ~0.88-0.90
(BERT-Klasse). Abstand: ~0.05-0.07 PR-AUC-Punkte. Konsistent
mit dem F1-Abstand — keine Ueberraschung, da PR-AUC und F1 eng
korrelieren bei binaerem Klassifikationsproblem.

In [ ]:
# =============================================================================
# Zelle 15 – keyword als zusaetzliches Feature (ColumnTransformer)
# =============================================================================
# keyword ist ein kuriertes Feld mit hoher Trennkraft (Notebook 01:
# Disaster-Rate 0%-100% je keyword, Kardinalitaets-Ratio 0.029).
# Hier: keyword als One-Hot-encodiertes kategoriales Feature zur
# TF-IDF-Matrix hinzufuegen via ColumnTransformer + Pipeline.

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import scipy.sparse as sp

# keyword: fehlende Werte mit "unknown" auffuellen
train["keyword_filled"] = train["keyword"].fillna("unknown")

print("Anzahl eindeutiger keywords:", train["keyword_filled"].nunique())
print("Fehlende keyword-Werte ersetzt:", (train["keyword"] == "unknown").sum())

# Manuelle Kombination: TF-IDF (Config C, Stemmed) + keyword OHE
# da ColumnTransformer mit sparse Matrizen arbeitet
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
keyword_encoded = ohe.fit_transform(
    train["keyword_filled"].values.reshape(-1, 1)
)

print(f"\nKeyword OHE Shape: {keyword_encoded.shape}")
print(f"TF-IDF Config C Shape: {configs['C'].shape}")

# Kombinierte Matrix: TF-IDF + keyword OHE horizontal stapeln
X_with_keyword = sp.hstack([configs["C"], keyword_encoded])
print(f"Kombinierte Matrix Shape: {X_with_keyword.shape}")

# Vergleich: mit vs. ohne keyword
model_kw = LogisticRegression(
    max_iter=1000, random_state=42, class_weight="balanced")

print("\n=== Vergleich: LogReg_balanced Config C ===")
print("OHNE keyword:")
result_no_kw = evaluate_model(
    model_kw, configs["C"], y, "LogReg_balanced", "C_no_keyword")
print_result(result_no_kw)

print("\nMIT keyword:")
result_with_kw = evaluate_model(
    model_kw, X_with_keyword, y, "LogReg_balanced", "C_with_keyword")
print_result(result_with_kw)

# Delta
delta_f1 = result_with_kw["f1_macro_mean"] - result_no_kw["f1_macro_mean"]
delta_roc = result_with_kw["roc_auc_mean"] - result_no_kw["roc_auc_mean"]
print(f"\n=== Delta (mit - ohne keyword) ===")
print(f"  F1 Macro:  {delta_f1:+.4f}")
print(f"  ROC-AUC:   {delta_roc:+.4f}")

# Export
kw_comparison = pd.DataFrame([result_no_kw, result_with_kw])
kw_comparison.to_csv(
    "../reports/tables/04_keyword_feature_comparison.csv", index=False)

## Erkenntnis: keyword als zusaetzliches Feature

**Ergebnis:** keyword verschlechtert die Performance minimal und
destabilisiert das Modell erheblich.

| Metrik | Ohne keyword | Mit keyword | Delta |
|---|---|---|---|
| F1 Macro | **0.7775 ± 0.0069** | 0.7746 ± 0.0200 | -0.0028 |
| ROC-AUC | **0.8487 ± 0.0122** | 0.8462 ± 0.0183 | -0.0025 |
| MCC | **0.5551 ± 0.0138** | 0.5495 ± 0.0400 | -0.0056 |

**Wichtigster Befund: Stabilitaetsverlust**
Die Standardabweichung verdoppelt sich fast (F1: ±0.007 → ±0.020,
MCC: ±0.014 → ±0.040).

**Erklaerung — datengetrieben verifiziert:**

1. **keyword ist redundant mit dem Fliesstext (88% Ueberlappung):**
   Bei 88% aller Tweets kommt das keyword-Wort bereits im Tweettext
   vor — TF-IDF auf dem Fliesstext erfasst dieses Signal bereits
   direkt. Das keyword als separate OHE-Spalte liefert kaum neue
   Information.

2. **keyword-Woerter sind bereits Top-TF-IDF-Features:**
   "fire" (Chi2=148), "bomb" (Chi2=119), "typhoon" (Chi2=71) — die
   wichtigsten keyword-Begriffe sind ohnehin die staerksten
   Trennwoerter im TF-IDF-Vokabular. Das Modell hat das Signal
   bereits, ohne explizites keyword-Feature.

3. **Warum die EDA-Signale taeuschen konnten:**
   In Notebook 01 zeigte keyword starke Disaster-Raten (0%-100%).
   Das Signal existiert — aber es steckt bereits im Fliesstext,
   nicht zusaetzlich im keyword-Feld. OHE auf keyword fuegt daher
   222 weitgehend redundante Spalten hinzu, die das Modell
   destabilisieren ohne echten Informationsgewinn.

**Konsequenz:** keyword wird NICHT als Feature in Phase 5 eingebaut.
Entscheidung beruht auf datengetriebener Verifikation (88%
Fliesstext-Ueberlappung, Chi2-Analyse), nicht auf Annahme.

**Offener Punkt:** Eine Disaster-Rate-Kodierung (keyword → numerischer
Wert 0.0-1.0 aus Notebook 01) statt OHE koennte das High-Redundanz-
Problem umgehen und wird als optionaler Tuning-Kandidat notiert.

In [ ]:
# =============================================================================
# Zelle 17 – Beste Modelle speichern (fuer Phase 5 + Phase 6)
# =============================================================================
# Gespeichert werden:
# - LogReg_balanced Config C (Hauptkandidat Phase 5)
# - MultinomialNB Config C (Vergleichskandidat)
# Beide werden auf dem GESAMTEN Trainingsdatensatz trainiert (kein Split)
# da Phase 5 (Tuning) und Phase 6 (Fehleranalyse) den finalen, voll
# trainierten Zustand benoetigen.

# Hauptkandidat: LogReg_balanced Config C
best_logreg = LogisticRegression(
    max_iter=1000, random_state=42, class_weight="balanced")
best_logreg.fit(configs["C"], y)
joblib.dump(best_logreg, "../models/baseline_logreg_balanced_C.joblib")

# Vergleichskandidat: MultinomialNB Config C
best_nb = MultinomialNB()
best_nb.fit(configs["C"], y)
joblib.dump(best_nb, "../models/baseline_nb_C.joblib")

# LinearSVC Config C (fuer Phase 5 Tuning — groesstes Tuning-Potenzial)
best_svc = LinearSVC(max_iter=2000, random_state=42)
best_svc.fit(configs["C"], y)
joblib.dump(best_svc, "../models/baseline_svc_C.joblib")

print("Gespeichert in models/:")
print("  baseline_logreg_balanced_C.joblib  (Hauptkandidat)")
print("  baseline_nb_C.joblib               (Vergleichskandidat)")
print("  baseline_svc_C.joblib              (Tuning-Kandidat Phase 5)")

# Registry aktualisieren
registry_update = """
## Baseline-Modelle (Phase 4)

| Datei | Modell | Config | F1 CV | ROC-AUC CV | Gap |
|---|---|---|---|---|---|
| baseline_logreg_balanced_C.joblib | LogReg balanced | C | 0.7775 | 0.8487 | 0.099 |
| baseline_nb_C.joblib | MultinomialNB | C | 0.7799 | 0.8446 | 0.077 |
| baseline_svc_C.joblib | LinearSVC | C | 0.7681 | 0.8302 | 0.181 |
"""

with open("../models/registry.md", "a") as f:
    f.write(registry_update)
print("\nregistry.md aktualisiert")

# Konfidenzintervalle der beiden Hauptkandidaten
print("\n=== Finale Punkt-Schaetzungen mit 95% KI ===")
for name, result in [
    ("LogReg_balanced Config C", result_no_kw),
    ("MultinomialNB Config C",
     nb_results[[r for r in range(len(nb_results))
                 if nb_results[r]["model"] == "MultinomialNB"
                 and nb_results[r]["config"] == "C"][0]])
]:
    print(f"\n{name}:")
    print(f"  F1 Macro:  {result['f1_macro_mean']:.4f} "
          f"[95% KI: {result['f1_macro_ci_low']:.3f} – "
          f"{result['f1_macro_ci_high']:.3f}]")
    print(f"  ROC-AUC:   {result['roc_auc_mean']:.4f} "
          f"[95% KI: {result['roc_auc_ci_low']:.3f} – "
          f"{result['roc_auc_ci_high']:.3f}]")
    print(f"  MCC:       {result['mcc_mean']:.4f}")
    print(f"  Gap:       {result['overfitting_gap']:.4f}")

# Abschluss: Notebook 04 — Baseline-Modelle & Ablationsstudie

## Datensatz-Status
- **Eingelesen:** `data/processed/train_preprocessed.csv` (6.801 Zeilen)
- **Modelle gespeichert:** 3 Baseline-Modelle in `models/`

## Ablationsplan — Ergebnisse

| Config | Methode | Dim | Bestes Modell | F1 CV | ROC-AUC |
|---|---|---|---|---|---|
| A | TF-IDF Unigram | 5.838 | LogReg_balanced | 0.7776 | 0.8464 |
| B | TF-IDF Bigram | 9.824 | LogReg_balanced | 0.7771 | 0.8449 |
| **C** | **TF-IDF Stemmed** | **4.869** | **MultinomialNB** | **0.7799** | 0.8446 |
| D | TF-IDF+LSA | 100 | LogReg | 0.7506 | 0.8134 |

**Config C (Stemmed TF-IDF) ist konsistent beste Vektorisierung**
in allen drei Modell-Typen — kleineres Vokabular durch Stammform-
Reduktion liefert besser generalisierbares Signal.

## Finale Kandidaten Phase 5

**Hauptkandidat:** LogReg_balanced Config C
  F1: 0.7775 ± 0.0069 [95% KI: 0.768 – 0.788]
  ROC-AUC: 0.8487 ± 0.0122 [95% KI: 0.840 – 0.859]
  MCC: 0.5551 | Gap: 0.099

**Vergleichskandidat:** MultinomialNB Config C
  F1: 0.7799 ± 0.0100 [95% KI: 0.770 – 0.789]
  ROC-AUC: 0.8446 ± 0.0081 [95% KI: 0.836 – 0.856]
  MCC: 0.5757 | Gap: 0.077

**Entscheidungsgrundlage:** F1-Differenz (0.0024) statistisch nicht
signifikant (innerhalb beider Standardabweichungen). LogReg bevorzugt
wegen niedrigerer Varianz, hoeherer ROC-AUC, besserer
Interpretierbarkeit und mehr Tuning-Hebeln.

## Kernerkenntnisse

1. **Alle Modelle in enger Bandbreite:** F1 0.750-0.780 — kein Modell
   dominiert klar. Das ist typisch fuer TF-IDF-basierte Klassifikation
   auf kurzen, kontextreichen Texten (Twitter-Daten).

2. **Fehler-Profile unterscheiden sich systemastisch:**
   NB: hohe Precision (0.825), niedriger Recall (0.636) — konservativ
   LogReg: ausgewogenes Profil (Precision 0.741, Recall 0.728)
   SVC: aehnlich LogReg, aber mit starkem Overfitting-Gap (0.18-0.22)

3. **LinearSVC: groesstes Tuning-Potenzial (Phase 5)**
   Train-F1 0.949 vs. CV-F1 0.768 — Gap 0.181 zeigt, dass starke
   Regularisierung (niedrigeres C) erhebliche Verbesserung ermoeglicht.

4. **keyword als Feature bringt keinen Mehrwert:**
   Datengetrieben bestaetigt: 88% Ueberlappung mit Fliesstext,